# `csk_dfe` — a chave CSK-DFE completa

Este notebook demonstra a biblioteca `csk_dfe` inteira: a composição e a decomposição da chave de 64 bits, o hash de segmentação do CNPJ e a representação Base62.

Para a resolução de tipos de documento (`TpDoc`) e a reversão de 7 bits, veja `notebooks/tpdoc.ipynb` — este notebook usa `TpDoc` como um componente já resolvido, sem repetir aquela demonstração.

É documentação executável, sem asserções: a verificação do comportamento é responsabilidade da suíte `pytest` em `tests/`.

In [1]:
from datetime import date, datetime

from csk_dfe import (
    Base62InvalidoError,
    ChaveInvalidaError,
    CnpjInvalidoError,
    DataInvalidaError,
    TpDoc,
    decode,
    from_base62,
    generate,
    hash_cnpj,
    to_base62,
)

## A chave, campo a campo

A chave de 64 bits reúne quatro campos: `AAMMDD * 2**43 + reverso * 2**36 + segmento * 2**30 + random_number`. O bit 63 é sempre `0`, garantindo que a chave seja representável como `BIGINT` positivo.

In [2]:
tpdoc = TpDoc.from_cod(5)
cnpj = "11.111.111/0001-91"
csk = generate(date(2022, 1, 1), tpdoc, cnpj)

d = decode(csk)
aammdd = int(d.dhemi.strftime("%y%m%d"))
reverso = d.tpdoc.get_reverse_cod()
segmento = d.hash_cnpj
random_number = d.random_number

print(f"csk = {csk} = 0b{csk:064b}")
print()
print(f"{'campo':<12}{'largura':>8}{'deslocamento':>14}{'valor':>10}   binário alinhado sob o deslocamento")
print(f"{'sinal':<12}{1:>8}{63:>14}{0:>10}   {0:01b}{'':>63}")
print(f"{'data':<12}{20:>8}{43:>14}{aammdd:>10}   {'':>1}{aammdd:020b}{'':>43}")
print(f"{'documento':<12}{7:>8}{36:>14}{reverso:>10}   {'':>21}{reverso:07b}{'':>36}")
print(f"{'cnpj base':<12}{6:>8}{30:>14}{segmento:>10}   {'':>28}{segmento:06b}{'':>30}")
print(f"{'random':<12}{30:>8}{0:>14}{random_number:>10}   {'':>34}{random_number:030b}")

csk = 1936034382482008864 = 0b0001101011011110001011010000001101101000110010000110101100100000

campo        largura  deslocamento     valor   binário alinhado sob o deslocamento
sinal              1            63         0   0                                                               
data              20            43    220101    00110101101111000101                                           
documento          7            36        80                        1010000                                    
cnpj base          6            30        13                               001101                              
random            30             0 684223264                                     101000110010000110101100100000


## `generate()` — as três formas de `dhemi`

A data de emissão aceita `datetime`, `date` ou string `AAMMDD`. A hora, quando presente, é descartada — a chave particiona por dia, não por instante.

In [3]:
csk_datetime = generate(datetime(2022, 1, 1, 23, 59, 59), tpdoc, cnpj)
csk_date = generate(date(2022, 1, 1), tpdoc, cnpj)
csk_string = generate("220101", tpdoc, cnpj)

campo_data = lambda csk: (csk >> 43) & 0xFFFFF

print(f"datetime (23:59:59) -> campo de data {campo_data(csk_datetime)}")
print(f"date                -> campo de data {campo_data(csk_date)}")
print(f"string '220101'     -> campo de data {campo_data(csk_string)}")
print()
print("as três formas produzem o mesmo campo de data: a hora é descartada.")

datetime (23:59:59) -> campo de data 220101
date                -> campo de data 220101
string '220101'     -> campo de data 220101

as três formas produzem o mesmo campo de data: a hora é descartada.


## `decode()` — os campos nomeados, e o CNPJ que não volta

`decode()` devolve uma `CskDecodificado` com `dhemi`, `tpdoc`, `hash_cnpj` e `random_number`. O CNPJ original não está em nenhum campo — só o segmento de 0 a 63 que ele produziu.

In [4]:
d = decode(csk)
print(d)
print()
print(f"dhemi         = {d.dhemi}")
print(f"tpdoc         = {d.tpdoc.get_name()} (código {d.tpdoc.get_cod()}, reverso {d.tpdoc.get_reverse_cod()})")
print(f"hash_cnpj     = {d.hash_cnpj}")
print(f"random_number = {d.random_number}")
print()
print(f"o CNPJ original era {cnpj!r} — decode() nunca devolve isso, só o segmento {d.hash_cnpj}.")

CskDecodificado(dhemi=datetime.date(2022, 1, 1), tpdoc=TpDoc(codigo=5, reverso=80, nome='NFCe'), hash_cnpj=13, random_number=684223264)

dhemi         = 2022-01-01
tpdoc         = NFCe (código 5, reverso 80)
hash_cnpj     = 13
random_number = 684223264

o CNPJ original era '11.111.111/0001-91' — decode() nunca devolve isso, só o segmento 13.


## Faixa SQL de um período

Como a data é o decimal literal `AAMMDD` sem *epoch*, um período de datas vira uma faixa contínua de valores da própria chave — consultável direto em SQL, sem decodificar linha a linha.

In [5]:
limite_inferior = 220101 * 2**43
limite_superior = 230801 * 2**43

print(f"documentos emitidos entre 01/01/2022 e 31/07/2023 (exclusive 01/08/2023):")
print(f"  limite inferior = 220101 * 2**43 = {limite_inferior}")
print(f"  limite superior = 230801 * 2**43 = {limite_superior}")
print()
print("em SQL: WHERE csk >= 220101 * POWER(2, 43) AND csk < 230801 * POWER(2, 43)")

documentos emitidos entre 01/01/2022 e 31/07/2023 (exclusive 01/08/2023):
  limite inferior = 220101 * 2**43 = 1936028870281003008
  limite superior = 230801 * 2**43 = 2030147065618628608

em SQL: WHERE csk >= 220101 * POWER(2, 43) AND csk < 230801 * POWER(2, 43)


## `hash_cnpj()` — filiais do mesmo contribuinte no mesmo segmento

O segmento é derivado só da raiz de 8 caracteres. CNPJ completo e raiz produzem o mesmo segmento, e é por isso que filiais de um mesmo contribuinte caem juntas.

In [6]:
raiz = "12345678"
matriz = raiz + "000191"
filial = raiz + "000272"

print(f"raiz   {raiz!r:>18} -> segmento {hash_cnpj(raiz)}")
print(f"matriz {matriz!r:>18} -> segmento {hash_cnpj(matriz)}")
print(f"filial {filial!r:>18} -> segmento {hash_cnpj(filial)}")
print()
print("os três caem no mesmo segmento: só os 8 primeiros caracteres importam.")

raiz           '12345678' -> segmento 13
matriz   '12345678000191' -> segmento 13
filial   '12345678000272' -> segmento 13

os três caem no mesmo segmento: só os 8 primeiros caracteres importam.


In [7]:
raizes = [f"{n:08d}" for n in range(0, 6400, 100)]
segmentos = [hash_cnpj(r) for r in raizes]

distribuicao = {s: segmentos.count(s) for s in sorted(set(segmentos))}
print(f"{len(raizes)} raízes distribuídas em {len(distribuicao)} segmentos distintos, de 0 a 63:")
print(sorted(distribuicao.keys()))

64 raízes distribuídas em 44 segmentos distintos, de 0 a 63:
[0, 1, 2, 3, 6, 8, 10, 11, 12, 13, 14, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 31, 32, 33, 34, 35, 36, 37, 39, 41, 43, 44, 45, 46, 47, 49, 54, 55, 56, 57, 59, 60, 61, 62]


## `random_number` — chaves diferentes, sem sequência

Duas chamadas de `generate()` com os mesmos argumentos produzem chaves diferentes. Os 30 bits menos significativos dão 2³⁰ ≈ 1,07 bilhão de combinações por dia, tipo de documento e segmento de CNPJ — a ordem de grandeza que o consumidor tem para dimensionar o próprio controle de colisão, já que a biblioteca não garante unicidade.

In [8]:
chaves = [generate(date(2022, 1, 1), tpdoc, cnpj) for _ in range(5)]
for k in chaves:
    print(k, "random_number =", decode(k).random_number)

print()
print(f"chaves distintas entre si: {len(set(chaves)) == len(chaves)}")
print(f"combinações por dia/tipo/segmento: 2**30 = {2**30:,}".replace(",", "."))

1936034382044066446 random_number = 246280846
1936034382186189223 random_number = 388403623
1936034382618926328 random_number = 821140728
1936034382041096967 random_number = 243311367
1936034382041742698 random_number = 243957098

chaves distintas entre si: True
combinações por dia/tipo/segmento: 2**30 = 1.073.741.824


## Base62 — texto curto que preserva a ordenação

`to_base62()` codifica a chave em 11 caracteres do alfabeto `0-9A-Za-z`, e a ordenação lexicográfica dos textos acompanha a ordenação numérica das chaves.

In [9]:
chaves_ordenadas = sorted(
    generate(d, tpdoc, cnpj)
    for d in (date(2022, 1, 1), date(2022, 6, 15), date(2023, 1, 1), date(2023, 8, 1))
)

for k in chaves_ordenadas:
    print(f"{k:>20}  ->  {to_base62(k)}")

textos = [to_base62(k) for k in chaves_ordenadas]
print()
print(f"textos já em ordem crescente: {textos == sorted(textos)}")
print(f"ida e volta: {[from_base62(t) for t in textos] == chaves_ordenadas}")

 1936034381946815459  ->  2J13azCnc0p
 1940555573660572335  ->  2JLlR6Ms0lT
 2023995312317173882  ->  2PVv2Ua7c6c
 2030152577733608091  ->  2Py7SbR2Dj1

textos já em ordem crescente: True
ida e volta: True


## Erros de domínio

Toda entrada inválida levanta uma exceção distinguível por tipo, todas derivadas de `CskDfeError` (por sua vez, de `ValueError`).

In [10]:
try:
    generate("220230", tpdoc, cnpj)
except DataInvalidaError as erro:
    print(f"DataInvalidaError: {erro}")

DataInvalidaError: data '220230' não é um dia real do calendário


In [11]:
try:
    hash_cnpj("1234567")
except CnpjInvalidoError as erro:
    print(f"CnpjInvalidoError: {erro}")

CnpjInvalidoError: CNPJ '1234567' tem menos de 8 caracteres alfanuméricos


In [12]:
try:
    decode(-1)
except ChaveInvalidaError as erro:
    print(f"ChaveInvalidaError: {erro}")

ChaveInvalidaError: valor -1 não é uma chave CSK-DFE válida de 63 bits


In [13]:
try:
    from_base62("texto-invalido")
except Base62InvalidoError as erro:
    print(f"Base62InvalidoError: {erro}")

Base62InvalidoError: texto Base62 'texto-invalido' não tem 11 caracteres
